# Topic 6: Recursion & Backtracking

**Goal**: Truly understand recursion, visualize the call stack, then master backtracking patterns.  
**Time**: ~6-8 hours  
**Prereqs**: Topics 0-4

---

## Why Is Recursion Hard?

Recursion feels unnatural because our brains think in loops — "do this, then this, then this." Recursion says "solve a smaller version of the same problem and trust it works."

**The trick**: don't try to trace every call in your head. Instead, think about:
1. **Base case**: when do I stop?
2. **Recursive case**: how do I make the problem smaller?
3. **Trust**: assume the recursive call gives the right answer for the smaller problem.

**Analogy**: Russian nesting dolls (matryoshka). To count all the dolls, open one → count the rest → add 1.

---

## Part 1: Understanding Recursion

## The Anatomy of Recursion

Every recursive function has exactly two parts:

```
def solve(problem):
    if problem is simple enough:    ← BASE CASE (stop here)
        return answer directly
    
    smaller = make problem smaller
    result = solve(smaller)         ← RECURSIVE CASE (call yourself)
    return combine(result)
```

If you forget the base case → infinite recursion → stack overflow!

```
Without base case:

solve(5) calls solve(4) calls solve(3) calls solve(2) calls ...
... calls solve(-999) calls solve(-1000) ... 💥 STACK OVERFLOW

With base case:

solve(5) calls solve(4) calls solve(3) calls solve(2) calls solve(1)
                                                              ↑
                                                         BASE CASE!
                                                         Stop here,
                                                         return answer.
```

## Visualizing the Call Stack

When a function calls itself, each call gets its own "frame" on the call stack — its own copy of local variables:

```
Call Stack (grows upward):

┌─────────────────────────┐
│ factorial(1) → return 1  │  ← base case reached, start returning
├─────────────────────────┤
│ factorial(2) → 2 * ?     │  ← waiting for factorial(1)
├─────────────────────────┤
│ factorial(3) → 3 * ?     │  ← waiting for factorial(2)
├─────────────────────────┤
│ factorial(4) → 4 * ?     │  ← waiting for factorial(3)
└─────────────────────────┘

Returns unwind (bottom-up):
  factorial(1) = 1
  factorial(2) = 2 * 1 = 2
  factorial(3) = 3 * 2 = 6
  factorial(4) = 4 * 6 = 24
```

Key insight: each frame is **frozen** while it waits for the call above it. When that call returns, the frozen frame **resumes** with the returned value.

In [ ]:
# === Factorial: The Simplest Recursive Function ===

def factorial_iterative(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


def factorial_recursive(n):
    if n <= 1:
        return 1
    return n * factorial_recursive(n - 1)


def factorial_traced(n, depth=0):
    """Shows exactly what happens at each recursion level."""
    indent = "  " * depth
    print(f"{indent}factorial({n})")
    
    if n <= 1:
        print(f"{indent}→ returning 1")
        return 1
    
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}→ returning {n} * {result // n} = {result}")
    return result


print("Iterative:", factorial_iterative(4))
print("Recursive:", factorial_recursive(4))
print()
print("=== Traced execution ===")
result = factorial_traced(4)
print(f"\nFinal answer: {result}")

## Example 2: Fibonacci

The classic recursive example — but also a lesson in WHY naive recursion can be terrible.

```
fib(5)
├── fib(4)
│   ├── fib(3)
│   │   ├── fib(2) → 1
│   │   └── fib(1) → 1
│   └── fib(2) → 1
└── fib(3)                 ← fib(3) computed AGAIN!
    ├── fib(2) → 1
    └── fib(1) → 1
```

This is O(2^n) — exponentially slow! `fib(50)` would take **years**.

The problem: **overlapping subproblems**. We compute `fib(3)` multiple times.

The fix: **memoization** — remember results we've already computed.

```
Without memo:              With memo:

fib(5) calls:              fib(5) calls:
  fib(4), fib(3)             fib(4), fib(3) ← cache hit!
  fib(3), fib(2)             fib(3), fib(2) ← cache hit!
  fib(2), fib(1)             fib(2), fib(1)
  fib(2), fib(1)             (done — only 5 unique calls)
  fib(1), fib(0)
  ... 15 total calls         vs 5 calls!
```

In [ ]:
# === Fibonacci: Naive vs Memoized ===

call_count = 0

def fib_naive(n):
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


def fib_memo(n, memo=None):
    global call_count
    call_count += 1
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)


print("=== Call count comparison ===")
for test_n in [10, 20, 30]:
    call_count = 0
    result_naive = fib_naive(test_n)
    naive_calls = call_count
    
    call_count = 0
    result_memo = fib_memo(test_n)
    memo_calls = call_count
    
    print(f"fib({test_n}) = {result_naive}")
    print(f"  Naive: {naive_calls:,} calls")
    print(f"  Memo:  {memo_calls} calls")
    print(f"  Speedup: {naive_calls / memo_calls:.0f}x")
    print()

## The Three Questions to Ask for Every Recursive Problem

Before writing any recursive function, answer these:

```
┌─────────────────────────────────────────────────────────┐
│  1. What is the BASE CASE?                              │
│     → When is the answer obvious / trivial?             │
│     → What's the smallest input I might receive?        │
│                                                         │
│  2. What is the RECURSIVE STEP?                         │
│     → How do I make the problem SMALLER?                │
│     → What recursive call do I make?                    │
│                                                         │
│  3. What do I do with the RESULT?                       │
│     → How do I combine the sub-result with current?     │
│     → Do I add to it? Multiply? Append?                 │
└─────────────────────────────────────────────────────────┘
```

| Problem | Base Case | Recursive Step | Combine |
|---------|-----------|----------------|----------|
| factorial(n) | n ≤ 1 → 1 | factorial(n-1) | n * result |
| sum(list) | empty → 0 | sum(list[1:]) | list[0] + result |
| reverse(str) | len ≤ 1 → str | reverse(str[1:]) | result + str[0] |
| power(x, n) | n = 0 → 1 | power(x, n-1) | x * result |

---

## Part 2: Classic Recursion Problems

## Problem 1: Sum of a List

The simplest possible recursive problem on a list.

```
Three questions:
  1. Base case: empty list → 0
  2. Recursive step: sum of the REST of the list
  3. Combine: first element + sum(rest)

Trace for [3, 1, 4, 1, 5]:

sum([3, 1, 4, 1, 5])
  = 3 + sum([1, 4, 1, 5])
  = 3 + (1 + sum([4, 1, 5]))
  = 3 + (1 + (4 + sum([1, 5])))
  = 3 + (1 + (4 + (1 + sum([5]))))
  = 3 + (1 + (4 + (1 + (5 + sum([])))))
  = 3 + (1 + (4 + (1 + (5 + 0))))
  = 14
```

In [ ]:
def recursive_sum(lst, depth=0):
    indent = "  " * depth
    print(f"{indent}sum({lst})")
    
    if not lst:
        print(f"{indent}→ base case: returning 0")
        return 0
    
    rest_sum = recursive_sum(lst[1:], depth + 1)
    result = lst[0] + rest_sum
    print(f"{indent}→ returning {lst[0]} + {rest_sum} = {result}")
    return result


print("=== Traced sum ===")
answer = recursive_sum([3, 1, 4, 1, 5])
print(f"\nFinal answer: {answer}")

## Problem 2: Reverse a String

```
Three questions:
  1. Base case: empty or single char → return it
  2. Recursive step: reverse everything EXCEPT the first char
  3. Combine: reversed_rest + first_char

Trace for "hello":

reverse("hello")
  = reverse("ello") + "h"
  = (reverse("llo") + "e") + "h"
  = ((reverse("lo") + "l") + "e") + "h"
  = (((reverse("o") + "l") + "l") + "e") + "h"
  = ((("o" + "l") + "l") + "e") + "h"
  = "olleh"
```

In [ ]:
def reverse_string(s, depth=0):
    indent = "  " * depth
    print(f'{indent}reverse("{s}")')
    
    if len(s) <= 1:
        print(f'{indent}→ base case: returning "{s}"')
        return s
    
    reversed_rest = reverse_string(s[1:], depth + 1)
    result = reversed_rest + s[0]
    print(f'{indent}→ returning "{reversed_rest}" + "{s[0]}" = "{result}"')
    return result


print("=== Traced reverse ===")
answer = reverse_string("hello")
print(f'\nFinal answer: "{answer}"')

## Problem 3: Power Function (x^n)

**Naive approach**: multiply x by itself n times → O(n).

**Fast approach** (exponentiation by squaring):
- If n is even: x^n = (x^(n/2))^2
- If n is odd:  x^n = x * x^(n-1)

This gives O(log n) — much faster for large exponents!

```
2^10 = (2^5)^2
2^5  = 2 * (2^4)
2^4  = (2^2)^2
2^2  = (2^1)^2
2^1  = 2 * (2^0)
2^0  = 1

Only 6 steps instead of 10!

For 2^1000: only ~10 steps instead of 1000!
```

In [ ]:
def power_naive(x, n, depth=0):
    """O(n) — one multiplication per level."""
    indent = "  " * depth
    print(f"{indent}power({x}, {n})")
    
    if n == 0:
        print(f"{indent}→ base case: returning 1")
        return 1
    
    result = x * power_naive(x, n - 1, depth + 1)
    print(f"{indent}→ returning {x} * {result // x} = {result}")
    return result


def power_fast(x, n, depth=0):
    """O(log n) — halve the exponent each time."""
    indent = "  " * depth
    print(f"{indent}power({x}, {n})")
    
    if n == 0:
        print(f"{indent}→ base case: returning 1")
        return 1
    
    if n % 2 == 0:
        half = power_fast(x, n // 2, depth + 1)
        result = half * half
        print(f"{indent}→ even: {half}^2 = {result}")
    else:
        result = x * power_fast(x, n - 1, depth + 1)
        print(f"{indent}→ odd: {x} * {result // x} = {result}")
    
    return result


print("=== Naive O(n) power ===")
print(f"Result: {power_naive(2, 5)}")

print("\n=== Fast O(log n) power ===")
print(f"Result: {power_fast(2, 10)}")

print("\n=== Step count comparison ===")
print("Naive 2^10: 10 recursive calls")
print("Fast  2^10: ~4 recursive calls (log2(10) ≈ 3.3)")

## Problem 4: Check if Array is Sorted

```
Three questions:
  1. Base case: 0 or 1 elements → always sorted (True)
  2. Recursive step: check if the REST is sorted
  3. Combine: first ≤ second AND rest_is_sorted

Trace for [1, 3, 5, 7]:

is_sorted([1, 3, 5, 7])
  1 ≤ 3? YES → check is_sorted([3, 5, 7])
    3 ≤ 5? YES → check is_sorted([5, 7])
      5 ≤ 7? YES → check is_sorted([7])
        base case: single element → True
      → True
    → True
  → True

Trace for [1, 5, 3, 7]:

is_sorted([1, 5, 3, 7])
  1 ≤ 5? YES → check is_sorted([5, 3, 7])
    5 ≤ 3? NO → return False immediately!
  → False
```

In [ ]:
def is_sorted(arr, depth=0):
    indent = "  " * depth
    print(f"{indent}is_sorted({arr})")
    
    if len(arr) <= 1:
        print(f"{indent}→ base case: True")
        return True
    
    if arr[0] > arr[1]:
        print(f"{indent}→ {arr[0]} > {arr[1]}: False!")
        return False
    
    print(f"{indent}  {arr[0]} ≤ {arr[1]} ✓")
    result = is_sorted(arr[1:], depth + 1)
    print(f"{indent}→ returning {result}")
    return result


print("=== Sorted array ===")
print(f"Result: {is_sorted([1, 3, 5, 7])}")

print("\n=== Unsorted array ===")
print(f"Result: {is_sorted([1, 5, 3, 7])}")

---

## Part 3: Backtracking — Recursion with Choices

Backtracking = recursion where you **explore choices**, and **undo** a choice if it doesn't work out.

**Analogy**: Navigating a maze. At each fork, pick a path. If you hit a dead end, **backtrack** to the fork and try a different path.

```
          START
            │
         ┌──┴──┐
         ▼     ▼
        [A]   [B]
        / \     |
       ▼   ▼    ▼
     DEAD  [C]  DEAD    ← hit dead end at A-left, backtrack!
      END   |    END    ← hit dead end at B, backtrack!
            ▼
          GOAL!         ← found solution via A → C
```

### The Backtracking Template

```
def backtrack(state, choices):
    if state is a valid solution:
        record the solution
        return
    
    for choice in choices:
        if choice is valid:
            make the choice          ← CHOOSE
            backtrack(new_state)     ← EXPLORE
            undo the choice          ← UN-CHOOSE (backtrack!)
```

The key insight: we BUILD a solution step by step. If adding something makes it invalid, we REMOVE it and try the next option. This is the **"choose → explore → un-choose"** pattern.

```
Pattern in action:

state = []         ← start empty

CHOOSE 'A':        state = ['A']
  EXPLORE deeper...
    CHOOSE 'B':    state = ['A', 'B']   ← dead end!
    UN-CHOOSE 'B': state = ['A']        ← backtrack!
    CHOOSE 'C':    state = ['A', 'C']   ← solution found!
  UN-CHOOSE 'A':   state = []           ← try other first choices

CHOOSE 'B':        state = ['B']
  EXPLORE deeper...
  ...
```

## Problem 5: Generate All Subsets

Given `[1, 2, 3]`, generate all subsets: `[], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]`.

The decision tree — for each element, we have 2 choices: **INCLUDE** it or **SKIP** it.

```
                         []
                    /          \
             include 1        skip 1
               /                  \
             [1]                   []
           /     \              /      \
      inc 2     skip 2     inc 2      skip 2
       /          \         /            \
    [1,2]         [1]     [2]            []
    /   \        /  \    /   \          / \
 inc3 skip3  inc3 sk3 inc3 skip3   inc3 skip3
  /     \    /    \    /     \      /     \
[1,2,3][1,2][1,3] [1] [2,3] [2]  [3]    []
```

Total subsets = 2^n (each element is either in or out).

In [ ]:
def subsets(nums):
    result = []
    
    def backtrack(index, current, depth=0):
        indent = "  " * depth
        
        if index == len(nums):
            print(f"{indent}→ LEAF: recording {current}")
            result.append(current[:])
            return
        
        # Choice 1: INCLUDE nums[index]
        print(f"{indent}Include {nums[index]}: {current} + [{nums[index]}]")
        current.append(nums[index])
        backtrack(index + 1, current, depth + 1)
        current.pop()  # UN-CHOOSE
        
        # Choice 2: SKIP nums[index]
        print(f"{indent}Skip {nums[index]}:    {current}")
        backtrack(index + 1, current, depth + 1)
    
    print("=== Decision tree for [1, 2, 3] ===")
    print()
    backtrack(0, [])
    return result


all_subsets = subsets([1, 2, 3])
print(f"\nAll subsets ({len(all_subsets)} total): {all_subsets}")

## Problem 6: Permutations

Given `[1, 2, 3]`, generate all orderings: `[1,2,3], [1,3,2], [2,1,3], [2,3,1], [3,1,2], [3,2,1]`.

```
                          []
                /          |          \
          choose 1     choose 2     choose 3
             /             |              \
           [1]            [2]            [3]
          / \            / \            / \
     ch 2   ch 3   ch 1   ch 3   ch 1   ch 2
      /       \      /       \      /       \
   [1,2]    [1,3] [2,1]   [2,3] [3,1]    [3,2]
    |         |     |       |     |         |
   ch 3     ch 2  ch 3   ch 1  ch 2      ch 1
    |         |     |       |     |         |
 [1,2,3] [1,3,2] [2,1,3] [2,3,1] [3,1,2] [3,2,1]
```

At each level, choose from **remaining** elements (not yet used).

Total permutations = n! (3! = 6 for three elements).

In [ ]:
def permutations(nums):
    result = []
    
    def backtrack(current, remaining, depth=0):
        indent = "  " * depth
        
        if not remaining:
            print(f"{indent}→ COMPLETE: {current}")
            result.append(current[:])
            return
        
        print(f"{indent}current={current}, choices={remaining}")
        
        for i in range(len(remaining)):
            choice = remaining[i]
            print(f"{indent}  pick {choice}")
            
            current.append(choice)
            new_remaining = remaining[:i] + remaining[i+1:]
            backtrack(current, new_remaining, depth + 1)
            current.pop()  # UN-CHOOSE
    
    print("=== Permutations of [1, 2, 3] ===")
    print()
    backtrack([], nums)
    return result


all_perms = permutations([1, 2, 3])
print(f"\nAll permutations ({len(all_perms)} total): {all_perms}")

## Problem 7: Combination Sum

Find all combinations of candidates that sum to target. You **CAN reuse** numbers.

```
candidates = [2, 3, 6, 7], target = 7
Output: [[2, 2, 3], [7]]

Decision tree (pruned):

                        target=7
                   /    |     |    \
               +2     +3    +6    +7
              /        |     |      \
          t=5        t=4   t=1    t=0 ✓ → [7]
         / | \       / \     |
       +2 +3 +6   +3 +6    ✗ (no candidate ≤ 1 except...)
       /    |  \    |
     t=3  t=2  ✗  t=1 → ✗
     /|    |
   +2 +3  +2
   /   \    \
 t=1   t=0✓ t=0 → ✗ (would give [2,3,2] = dup of [2,2,3])
  ✗   [2,2,3]
```

To avoid duplicates, we only consider candidates at index ≥ current index.

In [ ]:
def combination_sum(candidates, target):
    result = []
    candidates.sort()
    
    def backtrack(start, current, remaining, depth=0):
        indent = "  " * depth
        
        if remaining == 0:
            print(f"{indent}→ FOUND: {current} (sum = {target})")
            result.append(current[:])
            return
        
        if remaining < 0:
            print(f"{indent}→ OVERSHOT (remaining={remaining}), backtrack")
            return
        
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                print(f"{indent}→ {candidates[i]} > {remaining}, prune rest")
                break
            
            print(f"{indent}try +{candidates[i]}: {current + [candidates[i]]}, remaining={remaining - candidates[i]}")
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i], depth + 1)
            current.pop()
    
    print(f"=== Combination Sum: candidates={candidates}, target={target} ===")
    print()
    backtrack(0, [], target)
    return result


combos = combination_sum([2, 3, 6, 7], 7)
print(f"\nSolutions: {combos}")

## Problem 8: Generate Parentheses

Generate all valid combinations of `n` pairs of parentheses.

```
n = 3 → ["((()))", "(()())", "(())()", "()(())", "()()()"]
```

**Rules at any point**:
- Can add `(` if `open_count < n`
- Can add `)` if `close_count < open_count`

Decision tree for n=2:

```
                        ""
                        |
                       "("          ← can only open
                     /     \
                  "(("     "()"     ← can open or close
                   |       /  \
                 "(()"  "()(" "✗"   ← ")" invalid: close > open
                   |      |
                "(())"  "()()"
                  ✓       ✓
```

In [ ]:
def generate_parentheses(n):
    result = []
    
    def backtrack(current, open_count, close_count, depth=0):
        indent = "  " * depth
        print(f'{indent}build: "{current}" (open={open_count}, close={close_count})')
        
        if len(current) == 2 * n:
            print(f'{indent}→ VALID: "{current}"')
            result.append(current)
            return
        
        if open_count < n:
            backtrack(current + "(", open_count + 1, close_count, depth + 1)
        
        if close_count < open_count:
            backtrack(current + ")", open_count, close_count + 1, depth + 1)
    
    print(f"=== Generate Parentheses (n={n}) ===")
    print()
    backtrack("", 0, 0)
    return result


parens = generate_parentheses(3)
print(f"\nAll valid combinations: {parens}")
print(f"Count: {len(parens)}")

## Problem 9: N-Queens

Place N queens on an N×N board so no two queens threaten each other.

A queen can attack along rows, columns, and diagonals. Since we place one queen per row, we only need to check columns and diagonals.

Solution for N=4:
```
. Q . .        . . Q .
. . . Q        Q . . .
Q . . .        . . . Q
. . Q .        . Q . .

(2 solutions for N=4)
```

**Diagonal insight**:
```
For position (row, col):
  - Main diagonal (\): row - col is constant
  - Anti-diagonal (/): row + col is constant

    0   1   2   3
  ┌───┬───┬───┬───┐
0 │0-0│0-1│0-2│0-3│  row-col: 0, -1, -2, -3
  ├───┼───┼───┼───┤
1 │1-0│1-1│1-2│1-3│  row-col: 1,  0, -1, -2
  ├───┼───┼───┼───┤
2 │2-0│2-1│2-2│2-3│  row-col: 2,  1,  0, -1
  ├───┼───┼───┼───┤
3 │3-0│3-1│3-2│3-3│  row-col: 3,  2,  1,  0
  └───┴───┴───┴───┘

Same row-col → same \ diagonal
Same row+col → same / diagonal
```

**Algorithm**: Place one queen per row. For each row, try each column. Track which columns and diagonals are occupied.

In [ ]:
def solve_n_queens(n):
    solutions = []
    cols = set()
    diag1 = set()  # row - col
    diag2 = set()  # row + col
    board = [["."] * n for _ in range(n)]
    
    def print_board():
        for row in board:
            print("  " + " ".join(row))
        print()
    
    def backtrack(row):
        if row == n:
            solutions.append(["".join(r) for r in board])
            return
        
        for col in range(n):
            if col in cols or (row - col) in diag1 or (row + col) in diag2:
                continue
            
            # CHOOSE
            board[row][col] = "Q"
            cols.add(col)
            diag1.add(row - col)
            diag2.add(row + col)
            
            # EXPLORE
            backtrack(row + 1)
            
            # UN-CHOOSE
            board[row][col] = "."
            cols.remove(col)
            diag1.remove(row - col)
            diag2.remove(row + col)
    
    backtrack(0)
    return solutions


print("=== N-Queens (N=4) ===")
solutions = solve_n_queens(4)
print(f"Found {len(solutions)} solutions:\n")
for i, sol in enumerate(solutions):
    print(f"Solution {i + 1}:")
    for row in sol:
        print("  " + " ".join(row))
    print()

print("=== Solution counts for various N ===")
for board_size in range(1, 9):
    count = len(solve_n_queens(board_size))
    print(f"N={board_size}: {count} solution{'s' if count != 1 else ''}")

## Problem 10: Word Search

Given a 2D board and a word, check if the word exists by following adjacent cells (up/down/left/right). Each cell can only be used once per path.

```
board = [['A','B','C','E'],
         ['S','F','C','S'],
         ['A','D','E','E']]

"ABCCED" → True     (path: A→B→C→C→E→D)
"SEE"    → True     (path: S→E→E)
"ABCB"  → False    (can't reuse B)

Visualization of "ABCCED" path:

  [A] [B] [C]  E      1 → 2 → 3   .
   S   F  [C]  S      .   .   4   .
   A  [D] [E]  E      .   6 ← 5   .
```

**Algorithm**: For each cell matching the first letter, try to extend the path in all 4 directions. Mark cells as visited; unmark when backtracking.

In [ ]:
def word_search(board, word):
    rows, cols = len(board), len(board[0])
    
    def backtrack(r, c, index):
        if index == len(word):
            return True
        
        if (r < 0 or r >= rows or c < 0 or c >= cols or
                board[r][c] != word[index]):
            return False
        
        # CHOOSE: mark as visited
        temp = board[r][c]
        board[r][c] = "#"
        
        # EXPLORE: try all 4 directions
        found = (backtrack(r + 1, c, index + 1) or
                 backtrack(r - 1, c, index + 1) or
                 backtrack(r, c + 1, index + 1) or
                 backtrack(r, c - 1, index + 1))
        
        # UN-CHOOSE: restore cell
        board[r][c] = temp
        return found
    
    for r in range(rows):
        for c in range(cols):
            if board[r][c] == word[0] and backtrack(r, c, 0):
                return True
    return False


board = [['A','B','C','E'],
         ['S','F','C','S'],
         ['A','D','E','E']]

print("Board:")
for row in board:
    print(" ", " ".join(row))
print()

test_words = ["ABCCED", "SEE", "ABCB", "SFDE"]
for w in test_words:
    # Rebuild board each time since we mutate it
    board = [['A','B','C','E'],
             ['S','F','C','S'],
             ['A','D','E','E']]
    result = word_search(board, w)
    print(f'  "{w}" → {result}')

---

## Recursion vs Iteration — When to Use Which?

| Recursion | Iteration |
|---|---|
| Naturally fits tree/graph problems | Better for simple loops |
| Cleaner for divide-and-conquer | Uses less memory (no call stack) |
| Required for backtracking | Faster (no function call overhead) |
| Risk of stack overflow for deep recursion | No stack overflow risk |

**Rule of thumb**: if the problem has a **tree-like structure** (choices branching), use recursion. If it's a straight-line process, use a loop.

```
USE RECURSION when:              USE ITERATION when:

  Problem looks like a tree          Problem is linear
       *                             1 → 2 → 3 → 4 → 5
      / \
     *   *                           Simple accumulation
    / \ / \                          sum = 0
   *  * *  *                         for x in list:
                                         sum += x
  Backtracking needed
  Divide-and-conquer                 Performance critical
  Tree/graph traversal               (avoid call overhead)
```

---

## Practice Problems

| # | Problem | Difficulty | Key Pattern | LeetCode |
|---|---------|-----------|-------------|----------|
| 1 | Climbing Stairs | Easy | Base recursion + memo | #70 |
| 2 | Power of Two | Easy | Divide by 2 recursively | #231 |
| 3 | Merge Two Sorted Lists | Easy | Recursive merge | #21 |
| 4 | Subsets | Medium | Include/skip pattern | #78 |
| 5 | Permutations | Medium | Choose from remaining | #46 |
| 6 | Combination Sum | Medium | Backtrack with sum | #39 |
| 7 | Generate Parentheses | Medium | Constraint backtrack | #22 |
| 8 | Letter Combinations of Phone | Medium | Multi-choice backtrack | #17 |
| 9 | Word Search | Medium | Grid DFS + backtrack | #79 |
| 10 | N-Queens | Hard | Constraint checking | #51 |
| 11 | Sudoku Solver | Hard | Full backtracking | #37 |
| 12 | Palindrome Partitioning | Medium | Partition + check | #131 |

**Suggested order**: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 12 → 11

---

## Pattern Cheat Sheet

```
RECURSION & BACKTRACKING CHEAT SHEET:

"Simple recursion"            → Base case + recursive case + trust
"Overlapping subproblems"     → Add memoization (dict or @lru_cache)
"Generate all subsets"        → Include/skip each element
"Generate all permutations"   → Choose from remaining at each position
"Find combinations summing to k" → Backtrack with running sum
"Valid configurations"        → Backtrack + constraint checking (N-Queens)
"Path finding in grid"        → DFS + mark visited + unmark (backtrack)

THE TEMPLATE:
  def backtrack(state):
      if done: record solution
      for choice in options:
          if valid(choice):
              make(choice)
              backtrack(next_state)
              undo(choice)
```

```
COMPLEXITY GUIDE:

Problem                    Time           Space
─────────────────────────────────────────────────
Factorial / linear         O(n)           O(n) stack
Fibonacci (naive)          O(2^n)         O(n) stack
Fibonacci (memo)           O(n)           O(n)
Fast power                 O(log n)       O(log n) stack
Subsets                    O(2^n)         O(n) stack
Permutations               O(n!)          O(n) stack
N-Queens                   O(n!)          O(n)
```

```
DEBUGGING RECURSION:

1. Stack overflow?       → Check your base case. Is it reachable?
2. Wrong answer?         → Print state at each level (use depth param)
3. Duplicates?           → Are you exploring choices you shouldn't?
                           (use start index to skip previous elements)
4. Missing solutions?    → Are you pruning too aggressively?
5. Too slow?             → Look for overlapping subproblems → memoize
```

---

**Next up: Topic 7 — Sorting Algorithms**